In [1]:
# %pip install kagglehub
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("shaunthesheep/microsoft-catsvsdogs-dataset")

# print("Path to dataset files:", path)

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader,random_split
import torch.nn as nn 
import torch.optim as optim
import matplotlib.pyplot as pyplot
import torchvision
from torchvision import datasets,transforms as v2

In [3]:
# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# random seed for reproducibility
torch.manual_seed(42)

Using device: cuda


In [4]:
custom_transformations = v2.Compose([
    v2.Resize(256),
    v2.RandomHorizontalFlip(),
    v2.CenterCrop(224),
    v2.ToTensor(),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    

]
)

In [5]:
# CUSTom image folder class
from PIL import Image, UnidentifiedImageError
import os


class DogsandCatImageFolder(Dataset):
    def __init__(self,root_dir,transform):
        self.root_dir = root_dir
        self.transform = transform
        
        self.image_paths = []
        self.labels = []
        
        class_names = sorted(os.listdir(root_dir))
        
        self.class_to_idx = {
            class_name : idx
            for idx,class_name in enumerate(class_names)
        }
        
        for class_name in class_names:
            class_path = os.path.join(root_dir,class_name)
            for image_file_name in os.listdir(class_path):
                image_file_path = os.path.join(class_path,image_file_name)
                try:
                    with Image.open(image_file_path) as img:
                        img.verify()
                except(OSError, UnidentifiedImageError):
                    continue
                self.image_paths.append(image_file_path)
                self.labels.append(self.class_to_idx[class_name])
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, index):
        image_path = self.image_paths[index]
        label = self.labels[index]
        image = Image.open(image_path).convert("RGB")
        image = self.transform(image)
        return image,label

In [6]:
full_dataset = DogsandCatImageFolder(root_dir="PetImages",transform=custom_transformations)



C:\Users\shuva\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


In [7]:


# # image folder has its own get item internally so it can directly work liek dataset as its built on that but we r gonna do it maNUALLY
# full_dataset = datasets.ImageFolder(
#     root="PetImages",
#     transform=custom_transformations
# )

In [8]:
# from PIL import Image

# class CustomDataset(Dataset)):
#     def __init__(self,features,labels,transforms):
#         self.features = features
#         self.labels = labels
#         self.transforms = transforms
    
#     def __len__(self):
#         return len(self.features)
    
#     def __getitem__(self, index):
#         # 
#         return 
# # lol just realised i dont need to create a custom dataset, i just need to split

In [9]:
len(full_dataset)

24998

In [10]:
train_dataset,val_dataset,test_dataset = random_split(
    dataset=full_dataset,lengths=[19998,2500,2500]
)

In [11]:
train_loader = DataLoader(
    train_dataset,batch_size=32,shuffle=True,num_workers=0,pin_memory=True
)
validation_loader = DataLoader(
    val_dataset,batch_size=32,shuffle=False,
)
test_loader = DataLoader(
    test_dataset,batch_size=32,shuffle=False
)

In [12]:
# our input image is of 224x224 so we need to tweak a bit of padding

class MyNNAlexNet(nn.Module):
    def __init__(self):
        super().__init__()   
        self.featureExtractionLayer = nn.Sequential(
            nn.Conv2d(
                in_channels=3,
                out_channels=96,
                kernel_size=(11,11),
                stride=4,
                padding=2,
                ),
            nn.BatchNorm2d(96),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(3,3),stride=2),
            
            # 2nd layer
            
            nn.Conv2d(
                in_channels=96,
                out_channels=256,
                kernel_size=(5,5),
                padding=2
            ),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(3,3),stride=2),
            
            # 3rd layer
            
            nn.Conv2d(
                in_channels=256,
                out_channels=384,
                kernel_size=(3,3),
                padding=1,
            ),
            nn.BatchNorm2d(384),
            nn.ReLU(),
            
            # 4th layer
            
            nn.Conv2d(
                in_channels=384,
                out_channels=384,
                kernel_size=(3,3),
                padding=1
            ),
            nn.BatchNorm2d(384),
            nn.ReLU(),
                        
            # 5th layer
            nn.Conv2d(
                in_channels=384,
                out_channels=256,
                kernel_size=(3,3),
                padding=1
            ),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(stride=2,kernel_size=(3,3))
        )
        self.classifier = nn.Sequential(
            # 1st fc lauyer
            nn.Dropout(p=0.5),
            nn.Linear(9216,4096),
            nn.BatchNorm1d(4096),
            nn.ReLU(inplace=True),
            
            # 2nd fclayer
            nn.Dropout(p=0.5),
            nn.Linear(4096,4096),
            nn.BatchNorm1d(4096),
            nn.ReLU(inplace=True),
            # 3rd final layer
            
            nn.Linear(4096,1)
            
        )
    
    def forward(self,x):
        x = self.featureExtractionLayer(x)
        x = torch.flatten(x,1)
        x = self.classifier(x)
        return x

In [13]:
# for image,label in train_loader:
#     print(image)

In [14]:
model = MyNNAlexNet()
model.to(device)

MyNNAlexNet(
  (featureExtractionLayer): Sequential(
    (0): Conv2d(3, 96, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=(3, 3), stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(96, 256, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (5): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=(3, 3), stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(256, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(384, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Conv2d(384, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): BatchNorm2d(384, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (13): ReLU()
    (14): Conv2d(384, 256, kernel_si

In [15]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001,weight_decay=0.0001)

In [ ]:
epochs = 25 #optimal self tested
# train loop and eval l;oop combined
for epoch in range(epochs):

    model.train()
    total_epoch_loss = 0

    for images, labels in train_loader:

        images = images.to(device, non_blocking=True)
        labels = labels.float().to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images).squeeze(1)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_epoch_loss += loss.item()


    # Average training loss for this epoch
    average_loss = total_epoch_loss / len(train_loader)
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in validation_loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = model(images).squeeze(1)

            # Convert logits to probabilities
            probabilities = torch.sigmoid(outputs)
            predictions = (probabilities >= 0.5).long()
            correct += (predictions == labels).sum().item()
            total += labels.size(0)


    validation_accuracy = correct / total
    print(
        f"Epoch {epoch + 1}/{epochs}"
        f"Train Loss: {average_loss:.4f}"
        f"Validation Accuracy: {validation_accuracy * 100:.2f}%"
    )

Epoch 1/25Train Loss: 0.6945Validation Accuracy: 72.84%
Epoch 2/25Train Loss: 0.5262Validation Accuracy: 74.44%
Epoch 3/25Train Loss: 0.4233Validation Accuracy: 76.32%
Epoch 4/25Train Loss: 0.3572Validation Accuracy: 85.08%
Epoch 5/25Train Loss: 0.3154Validation Accuracy: 87.56%


In [ ]:
model.info()

NameError: name 'model' is not defined

In [ ]:
# eval loop
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images,labels in validation_loader:
        images = images.to(device)
        labels = labels.to(device)
        # predict
        outputs = model(images).squeeze(1)
        # find probabiltites
        probabilities = torch.sigmoid(outputs)
        prediction = (probabilities >= 0.5).long()
        
        correct += (prediction == labels).sum().item()
        total = total + labels.size()[0]

accuracy = correct/total
print(
        f"Validation Accuracy: {accuracy:.4f}"
    )

NameError: name 'model' is not defined

In [ ]:

model.eval()
dummy_input = torch.randn(1, 3, 224, 224).to(device)
Batch = 1
Channels = 3
Height = 224
Width = 224

torch.onnx.export(
    model,
    (dummy_input,),
    "alexNet.onnx",
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "output": {0: "batch_size"}
    }
)